# Nicheverse on seqFISH+

**Platform.** seqFISH+ (imaging based, high-plex; the classic NIH/3T3 fibroblast benchmark).

**Dataset (real).** seqFISH+ NIH/3T3 fibroblasts, prepared from the public seqFISH+ NIH/3T3 point-location release: **225 cells x
10,000 genes, 17 fields of view**. Per-cell counts were obtained by tallying the released
subcellular molecule point locations; raw integer counts are in `.X`, micron centroids in
`obsm['spatial']`, `obs['sample_id']` holds the FOV.

**Units.** `obsm['spatial']` is in **microns**.

**This dataset is tiny and homogeneous** (a single cultured fibroblast line), which changes
the recipe:

1. `cell_num_embeddings=256` cannot be filled by 225 cells, and the kmeans++ codebook
   initialization needs `n_cells >= cell_num_embeddings`. We therefore lower the codebook to
   **64 codes**.
2. The `mlp_plr` encoder over-parameterizes on 225 cells and collapses the codebook
   (measured: 1/256 codes). We use the plain **`mlp`** encoder here, which stays healthy on
   this size (measured: 61/64 codes active).

A tiny, homogeneous dataset cannot populate a large codebook. `mlp_plr`
with 256 codes is the right default for larger, diverse cohorts (Xenium / MERFISH), not for a
225-cell fibroblast benchmark. Real seqFISH+ tissue runs use ~300 epochs.


In [ ]:
import anndata as ad, numpy as np
PLATFORM = "seqFISH+ (NIH/3T3)"
adata = ad.read_h5ad("/path/to/your/seqfish_nih3t3.h5ad")  # point this at your prepared seqFISH+ AnnData
assert "spatial" in adata.obsm and "sample_id" in adata.obs
print(adata)
print("n_cells:", adata.n_obs, "| n_genes:", adata.n_vars,
      "| FOVs (samples):", adata.obs["sample_id"].nunique(),
      "| spatial units ~microns:", adata.obsm["spatial"].max(0).round(0))
# k_neighbors must not exceed the smallest FOV's cell count minus one
min_fov = int(adata.obs["sample_id"].value_counts().min())
k = max(2, min(8, min_fov - 1))
print("smallest FOV has", min_fov, "cells -> k_neighbors =", k)


## Configure and train

We build a `ModelConfig` (the architecture) and a `TrainConfig` (the optimization / spatial graph), then call `train_model`. The current library default encoder is `mlp_deep` (a SwiGLU pre-norm residual MLP) with the `vq` quantizer, and the neighborhood graph is `knn_radius` (radius 50 um, k = 20). We keep `batch_size=2048` rather than `'auto'`, because an over-large auto batch shrinks the number of optimizer steps per epoch and starves the codebook-diversity term. We run only a handful of demo epochs here so the notebook finishes in minutes; a production run uses about 300 epochs.

In [ ]:
import os
from nicheverse.models import ModelConfig, HierarchicalVQVAE
from nicheverse.training import train_model, TrainConfig

ckpt = "runs/nb_seqfish_demo"
os.makedirs(ckpt, exist_ok=True)

# Tiny dataset: lower codebook to 64 (225 cells cannot fill 256; kmeans++ init needs
# n_cells >= cell_num_embeddings), and use the plain mlp encoder because mlp_plr
# over-parameterizes and collapses to a single code on 225 cells (verified: mlp gives
# 61/64 codes active, mlp_plr gives 1/256).
mc = ModelConfig(
    input_dim=int(adata.n_vars),
    cell_embedding_dim=64, cell_num_embeddings=64,     # lowered: n_cells (225) < 256
    neighborhood_embedding_dim=256, neighborhood_num_embeddings=16,
    use_cross_attention=True,
    gene_names=tuple(adata.var_names.astype(str)),
    encoder_type="mlp", quantizer_type="vq",           # mlp_plr collapses on 225 cells
)
tc = TrainConfig(
    num_epochs=40,             # demo; production ~300
    batch_size=256,            # 225 cells fit in a single batch
    learning_rate=3e-4,
    spatial_graph="knn", k_neighbors=k,                # tiny FOVs: cap k, plain knn
    normalize=True, log1p=True, seed=9,   # default seed
)
model, adata = train_model(adata, ckpt, model_config=mc, train_config=tc, sample_col="sample_id")
print("done ->", ckpt)


## Inspect the learned codebook

`train_model` writes the per-cell code assignment to `hierarchical_cell_indices.npz` (key `indices`). A well-utilized codebook spreads cells across many codes; a collapsed run concentrates almost all cells in a few codes.

In [ ]:

# --- Load the codes the model just assigned to every cell ---
import numpy as np, json, os
idx = np.load(os.path.join(ckpt, "hierarchical_cell_indices.npz"))["indices"].ravel()
n_codes = int(model.config.cell_num_embeddings)
u, counts = np.unique(idx, return_counts=True)
print(f"{PLATFORM}: {len(idx)} cells assigned to {len(u)}/{n_codes} cell codes "
      f"(codebook usage {100*len(u)/n_codes:.0f}%)")


In [ ]:

# --- Code-usage bar chart (how many cells fall in each active code) ---
%matplotlib inline
import matplotlib.pyplot as plt
order = np.argsort(counts)[::-1]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(u)), counts[order], color="#3b6ea5")
ax.set_xlabel("cell code (sorted by usage)")
ax.set_ylabel("n cells")
ax.set_title(f"{PLATFORM}: cell-code usage ({len(u)}/{n_codes} codes active)")
plt.tight_layout()
plt.show()


## Top markers per code

For each used code we z-score its mean expression across codes and list the most enriched panel genes. This is a quick biological sanity check that codes track distinct cell states.

In [ ]:

# --- Per-code top-marker table: mean log1p expression per code, z-scored across codes ---
import pandas as pd, scanpy as sc
work = adata.copy()
sc.pp.normalize_total(work); sc.pp.log1p(work)
X = work.X.toarray() if hasattr(work.X, "toarray") else np.asarray(work.X)
genes = np.asarray(work.var_names)
rows = []
for c in u:                                   # only codes that are actually used
    m = X[idx == c].mean(0)
    rows.append(m)
M = np.vstack(rows)                            # (n_used_codes, n_genes)
Z = (M - M.mean(0)) / (M.std(0) + 1e-8)        # z across codes, per gene
topk = 6
recs = []
for r, c in enumerate(u):
    top = genes[np.argsort(Z[r])[::-1][:topk]]
    recs.append({"cell_code": int(c), "n_cells": int((idx == c).sum()),
                 "top_markers": ", ".join(top)})
marker_tbl = pd.DataFrame(recs).sort_values("n_cells", ascending=False).reset_index(drop=True)
print(f"Top {topk} enriched genes per used cell code (first 15 codes shown):")
marker_tbl.head(15)


## Training runtime

The trainer records wall-clock time, throughput (cells/sec), and peak GPU memory to `training_runtime.json`.

In [ ]:

# --- Training runtime report the trainer wrote (real timing on this GPU run) ---
rt_path = os.path.join(ckpt, "training_runtime.json")
runtime = json.load(open(rt_path))
print(json.dumps(runtime, indent=2))
